# فصل 11: پردازش ابری و کلان‌داده‌های مکانی

کد این نوت‌بوک عیناً از فصل 11 کتاب «مهندسی هوش مصنوعی مکانی» برداشته شده است.

پیش از اجرا: مسیر داده‌ها را با ساختار پوشه‌ی `data/`، `work/` و `out/` هماهنگ کنید. راه دانلود هر فایل در پیوست ب کتاب آمده است.


## ۱۱-۴) گام‌های اول با GEE در پایتون


In [ ]:
!pip install earthengine-api geemap

import ee, geemap

ee.Authenticate()                        # opens a one-time login URL
ee.Initialize(project="ee-yourname")     # your Google Cloud project

In [ ]:
tehran = ee.Geometry.Point([51.4, 35.7])
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

# 1) FILTER: pick only what we need
filtered = (s2.filterBounds(tehran)
            .filterDate("2023-01-01", "2023-12-31")
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))

# 2) MAP: add an NDVI band to every image (in parallel)
def add_ndvi(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

with_ndvi = filtered.map(add_ndvi)

# 3) REDUCE: mean NDVI per pixel across the whole year
ndvi_annual = with_ndvi.select("NDVI").mean()

## ۱۱-۵) کارگاه ۱: روند بیست‌ساله‌ی خشک‌سالی از NDVI


In [ ]:
fars = (ee.FeatureCollection("FAO/GAUL/2015/level1")
        .filter(ee.Filter.eq("ADM1_NAME", "Fars")))

modis = (ee.ImageCollection("MODIS/061/MOD13Q1")
         .filterDate("2004-01-01", "2024-12-31")
         .filterBounds(fars)
         .select("NDVI"))

def yearly_mean(y):
    y = ee.Number(y)
    start = ee.Date.fromYMD(y, 1, 1)
    return (modis.filterDate(start, start.advance(1, "year"))
            .mean().multiply(0.0001).set("year", y))

years = ee.List.sequence(2004, 2024)
yearly = ee.ImageCollection.fromImages(years.map(yearly_mean))

# slope of NDVI over time, per pixel  (negative = drying out)
trend = (yearly.reduce(ee.Reducer.linearFit())
         .select("scale").clip(fars))

## ۱۱-۶) کارگاه ۲: Random Forest روی ابر برای پوشش زمین


In [ ]:
roi = (ee.FeatureCollection("FAO/GAUL/2015/level1")
       .filter(ee.Filter.eq("ADM1_NAME", "Esfahan")))

composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterBounds(roi)
             .filterDate("2024-04-01", "2024-09-30")
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 15))
             .median()
             .clip(roi))

ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
composite = composite.addBands(ndvi)
bands = ["B2", "B3", "B4", "B8", "B11", "NDVI"]

# labels from an existing land-cover product (weak supervision)
wc = ee.Image("ESA/WorldCover/v200/2021").clip(roi)
samples = composite.select(bands).sampleRegions(
    collection=wc.sample(roi.geometry(), scale=100, numPixels=5000),
    properties=["Map"],
    scale=10)

clf = ee.Classifier.smileRandomForest(100).train(samples, "Map", bands)
classified = composite.select(bands).classify(clf).clip(roi)